In [2]:
import pandas as pd
import os
from itertools import permutations, combinations
import re
import numpy as np

In [3]:
import rasterio
import numpy as np
from datetime import datetime

In [4]:
path = "boyaUPCT"
filename = "locBoyasUPCT_reproyectado.csv"
loc_boyas = pd.read_csv(os.path.join(path, filename)).iloc[:,:5]

In [5]:
def extract_values_for_buoys(folder_path, loc_boyas, target_dates, grouping, net_set):

    """
    Extract pixel values from TIFF files for each buoy in loc_boyas and for each specified date.
    """
    results = []
    target_dates = sorted(set(target_dates))
    if net_set == "C2X-Complex":
        tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif') and 'C2XComplexNets' in f]
    elif net_set == "C2X":
        tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif') and 'C2XNets' in f]
    elif net_set == "C2RCC":
        tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif') and 'C2RCC' in f]
    else:
        print("Net Set no reconocido")

    for date in target_dates:
        date_to_find = date.replace('-', '')
        matching = [f for f in tif_files if date_to_find in f]
        if matching:
            tiff_file = os.path.join(folder_path, matching[0])  

            with rasterio.open(tiff_file) as dataset:
                print(f"Processing {tiff_file}")
                bands = dataset.read()

                for _, row in loc_boyas.iterrows():
                    buoy_id = row["CodPuntoControl"].replace('-', '').strip()
                    lat = int(round(row["LatitudEPSG32630"]))
                    lon = int(round(row["LongitudEPSG32630"]))

                    try:
                        row_idx, col_idx = dataset.index(lon, lat)

                        if grouping == "3x3":
                            offset = 1
                        elif grouping == "5x5":
                            offset = 2
                        elif grouping == "9x9":
                            offset = 4
                        elif grouping == "15x15":
                            offset = 7
                        else:
                            offset = 0

                        if offset > 0:
                            window = (
                                slice(max(row_idx - offset, 0), min(row_idx + offset + 1, dataset.height)),
                                slice(max(col_idx - offset, 0), min(col_idx + offset + 1, dataset.width))
                            )
                            reflectances = bands[:, window[0], window[1]]
                            values = np.median(reflectances, axis=(1, 2))
                        else:
                            values = bands[:, row_idx, col_idx]

                        results.append({
                            "Date": date,
                            "Buoy": buoy_id,
                            "Latitude": lat,
                            "Longitude": lon,
                            **{f"Band_{i+1}": val for i, val in enumerate(values)}
                        })

                    except IndexError:
                        print(f"Skipping {buoy_id} on {date}: Coordinates out of raster bounds")

    return pd.DataFrame(results)

In [10]:
folder_path = "Copernicus/SAFE_downloads/application"
target_dates = [
    '2016-09-08', '2022-07-14','2023-04-20'
]

groupings = ["5x5", "9x9"]
net_set = ["C2X-Complex"]
for grouping in groupings:
    for net in net_set:
        df_tiffs = extract_values_for_buoys(folder_path, loc_boyas, target_dates, grouping, net)
        df_tiffs["Date"] = pd.to_datetime(df_tiffs["Date"])
        df_tiffs.to_csv(f"saved_files/application/df_tifs_{net}_{grouping}.csv", index=False)

Processing Copernicus/SAFE_downloads/application/S2A_MSIL1C_20160908T105022_N0500_R051_T30SXG_20231030T134748_C2XComplexNets_10m.tif
Processing Copernicus/SAFE_downloads/application/S2B_MSIL1C_20220714T104629_N0510_R051_T30SXG_20240723T192911_C2XComplexNets_10m.tif
Processing Copernicus/SAFE_downloads/application/S2B_MSIL1C_20230420T104619_N0510_R051_T30SXG_20240903T235722_C2XComplexNets_10m.tif
Processing Copernicus/SAFE_downloads/application/S2A_MSIL1C_20160908T105022_N0500_R051_T30SXG_20231030T134748_C2XComplexNets_10m.tif
Processing Copernicus/SAFE_downloads/application/S2B_MSIL1C_20220714T104629_N0510_R051_T30SXG_20240723T192911_C2XComplexNets_10m.tif
Processing Copernicus/SAFE_downloads/application/S2B_MSIL1C_20230420T104619_N0510_R051_T30SXG_20240903T235722_C2XComplexNets_10m.tif


In [9]:
df_tiffs

,Date,Buoy,Latitude,Longitude,Band_1,Band_2,Band_3,Band_4,Band_5,Band_6,...,Band_13,Band_14,Band_15,Band_16,Band_17,Band_18,Band_19,Band_20,Band_21,Band_22
0,2016-09-08,CTD1,4187246,695025,0.1222,0.0924,0.0745,0.0498,0.0428,0.0348,...,0.0124,0.010580,0.015641,0.026116,0.013858,0.012464,0.003613,0.003531,0.001461,-8.407791e-45
1,2016-09-08,CTD2,4181518,693105,0.1193,0.0889,0.0689,0.0435,0.0371,0.0301,...,0.0090,0.010295,0.014946,0.024088,0.011801,0.009841,0.002800,0.002756,0.001132,-8.407791e-45
2,2016-09-08,CTD3,4181698,695238,0.1178,0.0865,0.0661,0.0406,0.0346,0.0275,...,0.0072,0.010014,0.014341,0.022994,0.011156,0.009591,0.002780,0.002722,0.001121,-8.407791e-45
3,2016-09-08,CTD4,4180266,698264,0.1196,0.0895,0.0668,0.0440,0.0375,0.0311,...,0.0107,0.009696,0.013779,0.020961,0.011210,0.009079,0.002640,0.002618,0.001096,-8.407791e-45
4,2016-09-08,CTD5,4179450,700268,0.1238,0.0950,0.0721,0.0403,0.0353,0.0322,...,0.0120,0.009379,0.014307,0.019780,0.004109,0.003065,0.000704,0.000716,0.000276,-2.878267e-42
5,2016-09-08,CTD6,4176009,695829,0.1166,0.0862,0.0644,0.0400,0.0337,0.0272,...,0.0070,0.008654,0.012531,0.019597,0.010282,0.008222,0.002336,0.002341,0.000964,-8.407791e-45
6,2016-09-08,CTD7,4176724,690397,0.1202,0.0895,0.0703,0.0444,0.0377,0.0308,...,0.0083,0.010933,0.016065,0.025683,0.012873,0.010826,0.003091,0.003051,0.001244,-8.407791e-45
7,2016-09-08,CTD8,4174178,693048,0.1192,0.0891,0.0699,0.0454,0.0388,0.0305,...,0.0088,0.009821,0.014591,0.024009,0.012915,0.011146,0.003214,0.003169,0.001307,-8.407791e-45
8,2016-09-08,CTD9,4171106,693183,0.1210,0.0933,0.0792,0.0541,0.0467,0.0355,...,0.0126,0.011150,0.017611,0.032063,0.018600,0.016469,0.004695,0.004692,0.001913,-8.407791e-45
9,2016-09-08,CTD10,4170388,695646,0.1201,0.0914,0.0751,0.0490,0.0424,0.0322,...,0.0101,0.011831,0.017778,0.031085,0.016911,0.014817,0.004297,0.004216,0.001726,-8.407791e-45


In [11]:
# Cargamos los csv de los tifs
path = "saved_files/application"
dfs_tifs = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_tifs_") and archivo.endswith(".csv") and "planet" not in archivo:
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs_tifs[nombre_sin_extension] = pd.read_csv(ruta_completa)

In [14]:
dfs_tifs["df_tifs_C2X-Complex_5x5"]

,Date,Buoy,Latitude,Longitude,Band_1,Band_2,Band_3,Band_4,Band_5,Band_6,...,Band_13,Band_14,Band_15,Band_16,Band_17,Band_18,Band_19,Band_20,Band_21,Band_22
0,2016-09-08,CTD1,4187246,695025,0.1222,0.0927,0.0743,0.0501,0.0442,0.0353,...,0.0124,0.011359,0.016524,0.027251,0.014605,0.013168,0.003859,0.003769,0.001570,-8.000000e-45
1,2016-09-08,CTD2,4181518,693105,0.1193,0.0889,0.0688,0.0436,0.0371,0.0301,...,0.0089,0.009683,0.014278,0.023085,0.011246,0.009418,0.002605,0.002522,0.001035,-8.000000e-45
2,2016-09-08,CTD3,4181698,695238,0.1178,0.0865,0.0658,0.0404,0.0343,0.0274,...,0.0071,0.010257,0.014695,0.023492,0.011051,0.009323,0.002677,0.002583,0.001067,-8.000000e-45
3,2016-09-08,CTD4,4180266,698264,0.1196,0.0897,0.0671,0.0441,0.0378,0.0310,...,0.0110,0.011195,0.015667,0.023484,0.013225,0.010400,0.003101,0.003005,0.001272,-8.000000e-45
4,2016-09-08,CTD5,4179450,700268,0.1238,0.0961,0.0734,0.0404,0.0357,0.0324,...,0.0120,0.010466,0.016054,0.021233,0.004241,0.003048,0.000704,0.000716,0.000276,-2.878000e-42
5,2016-09-08,CTD6,4176009,695829,0.1166,0.0863,0.0645,0.0403,0.0340,0.0272,...,0.0070,0.008070,0.011950,0.018958,0.009958,0.007988,0.002211,0.002189,0.000927,-8.000000e-45
6,2016-09-08,CTD7,4176724,690397,0.1202,0.0898,0.0703,0.0448,0.0377,0.0308,...,0.0083,0.010933,0.016185,0.026133,0.013000,0.010498,0.003003,0.002958,0.001221,-8.000000e-45
7,2016-09-08,CTD8,4174178,693048,0.1193,0.0891,0.0698,0.0454,0.0388,0.0309,...,0.0088,0.009807,0.014458,0.024009,0.012915,0.011194,0.003249,0.003143,0.001302,-8.000000e-45
8,2016-09-08,CTD9,4171106,693183,0.1213,0.0932,0.0792,0.0542,0.0465,0.0354,...,0.0126,0.011150,0.017658,0.032415,0.018647,0.016634,0.004722,0.004727,0.001919,-8.000000e-45
9,2016-09-08,CTD10,4170388,695646,0.1201,0.0913,0.0748,0.0487,0.0424,0.0324,...,0.0099,0.011680,0.017563,0.030291,0.016079,0.014313,0.004128,0.004106,0.001667,-8.000000e-45


In [31]:
band_names = {
    "Band_1": "rtoa_B1",
    "Band_2": "rtoa_B2",
    "Band_3": "rtoa_B3",
    "Band_4": "rtoa_B4",
    "Band_5": "rtoa_B5",
    "Band_6": "rtoa_B6",
    "Band_7": "rtoa_B7",
    "Band_8": "rtoa_B8",
    "Band_9": "rtoa_B8A",
    "Band_10": "rtoa_B9",
    "Band_11": "rtoa_B10",
    "Band_12": "rtoa_B11",
    "Band_13": "rtoa_B12",
    "Band_14": "rhow_B1",
    "Band_15": "rhow_B2",
    "Band_16": "rhow_B3",
    "Band_17": "rhow_B4",
    "Band_18": "rhow_B5",
    "Band_19": "rhow_B6",
    "Band_20": "rhow_B7",
    "Band_21": "rhow_B8A"
}

dfs_tifs_all = {} 

for nombre_df, df in list(dfs_tifs.items()):
    # Rename de todas la columnas
    df = df.rename(columns=band_names)

    # Para coger de solamente de C2RCC las bandas TOA, puesto que son iguales en C2X y Complex
    if "9x9" in nombre_df:
    #     #print(nombre_df)
        ventana = nombre_df.split("_")[3]
        dfs_tifs_all[f"df_tifs_TOA_{ventana}"] = df.iloc[:,np.r_[0:17]]


    name_chunks = nombre_df.split("_")
    # El resto de columnas igual
    dfs_tifs_all[f"{name_chunks[0]}_{name_chunks[1]}_{name_chunks[2]}_rhow_{name_chunks[3]}"] = df.iloc[:,np.r_[0:4, 17:25]]
    #dfs_tifs_all[f"{name_chunks[0]}_{name_chunks[1]}_{name_chunks[2]}_rhown_{name_chunks[3]}"] = df.iloc[:,np.r_[0:4,25:31]]
    #dfs_tifs[nombre_df] = df.rename(columns=band_names)

In [18]:
def diferencia_normalizada(band1, band2):
    value = (band1 - band2)/(band1 + band2)
    return value.round(3)

def dall_gitelson(band1, band2, band3):
    value = (1/(band1) - 1/(band2))*(band3)
    return value.round(3)

def diferencia_normalizada_4bandas(band1, band2, band3, band4):
    value = (band1 - band2)/(band3 + band4)
    return value.round(3)

def diferencia_inversas(band1, band2):
    value = 1/(band1) - 1/(band2)
    return value.round(3)

def diferencia_relacion_4bandas(band1, band2, band3, band4):
    value = band1/band2 - band3/band4
    return value.round(3)

def suma_normalizada_3bandas(band1, band2, band3):
    value = (band1 + band3)/(band1 + band2)
    return value.round(3)

def add_two_band_difs(data, bands):
    for i, band1 in enumerate(bands):
        for band2 in bands[i+1:]:
            colname_dif_norm = f"dif_norm_{band1}_{band2}"
            data[colname_dif_norm] = diferencia_normalizada(data[band1], data[band2])
            #index_list.append(colname_dif_norm)
            colname_dif_inv = f"dif_inv_{band1}_{band2}"
            data[colname_dif_inv] = diferencia_inversas(data[band1], data[band2])
            #index_list.append(colname_dif_inv)
    return data

def add_dall_gitelson(data, bands):
    for band1, band2 in combinations(bands, 2):  # evita repeticiones de pares
        for band3 in bands:
            if band3 not in (band1, band2):  # evitar que band3 sea igual a los anteriores
                colname = f"dall_gitelson_{band1}_{band2}_{band3}"
                data[colname] = dall_gitelson(data[band1], data[band2], data[band3])
                #index_dall_gitelson_list.append(colname)
    return data

def add_norm_dif_4bands(data, bands):
    for band1, band2 in combinations(bands, 2):  # evita invertir band1 y band2
        for band3, band4 in combinations(bands, 2):  # evita invertir band3 y band4
            # Asegurar que todas las bandas son distintas
            if len({band1, band2, band3, band4}) == 4:
                colname = f"dif_norm_4_bands_{band1}_{band2}_{band3}_{band4}"
                data[colname] = diferencia_normalizada_4bandas(
                    data[band1], data[band2], data[band3], data[band4]
                )
                #index_dif_norm_4bands_list.append(colname)
    return data

def add_index_dif_rel_4bands(data, bands):
    for band1, band2, band3, band4 in permutations(bands, 4):
        # Evitar redundancias por simetría de términos
        # Criterio: solo aceptamos combinaciones donde el primer término es "menor" que el segundo
        if (band1, band2) < (band3, band4):  # evita generar la versión espejo con signo opuesto
            colname = f"dif_rel_4bands_{band1}_{band2}_{band3}_{band4}"
            data[colname] = diferencia_relacion_4bandas(
                data[band1], data[band2], data[band3], data[band4]
            )
            #index_dif_rel_4bands_list.append(colname)
    return data

def add_index_sum_norm_3bands(data, bands):
    band1, band2, band3, band4 = bands

    colname = f"sum_norm_3bands_{band1}_{band3}_{band2}"
    data[colname] = suma_normalizada_3bandas(data[band1], data[band2], data[band3])

    colname = f"sum_norm_3bands_{band2}_{band4}_{band3}"
    data[colname] = suma_normalizada_3bandas(data[band2], data[band3], data[band4])
       
    return data

In [32]:
band_sets = [
    ['rtoa_B2', 'rtoa_B3', 'rtoa_B4', 'rtoa_B5'],
    ['rhow_B2', 'rhow_B3', 'rhow_B4', 'rhow_B5']
]

dfs = dfs_tifs_all.copy()
for nombre_df, df in dfs.items():
    for bands_to_use in band_sets:
        if set(bands_to_use).issubset(df.columns):
            df = add_two_band_difs(df, bands_to_use)
            df = add_dall_gitelson(df, bands_to_use)
            df = add_norm_dif_4bands(df, bands_to_use)
            df = add_index_dif_rel_4bands(df, bands_to_use)
            df = add_index_sum_norm_3bands(df, bands_to_use)
            
    dfs[nombre_df] = df 

In [33]:
import re

def compactar_prefijos_columnas(df):
    nuevo_nombre_columnas = {}

    for col in df.columns:
        # Detectar columnas con patrones tipo index_algo_rhow_B1_rhow_B2_...
        if re.search(r'(rhow|rhown|rtoa)(_B\d+)+', col):
            partes = col.split('_')
            base = []
            bandas = []
            prefijo = None

            for parte in partes:
                if parte in ['rhow','rtoa']:
                    if not prefijo:
                        prefijo = parte
                elif parte.startswith('B'):
                    bandas.append(parte)
                else:
                    base.append(parte)

            if prefijo and bandas:
                #nuevo_nombre = f"{'_'.join(base)}_{prefijo}_{'_'.join(bandas)}"
                if base:
                    nuevo_nombre = f"{'_'.join(base)}_{prefijo}_{'_'.join(bandas)}"
                else:
                    nuevo_nombre = f"{prefijo}_{'_'.join(bandas)}"

                nuevo_nombre_columnas[col] = nuevo_nombre

    # Renombrar columnas
    df = df.rename(columns=nuevo_nombre_columnas)
    return df

In [34]:
for nombre_df, df in dfs.items():
    dfs[nombre_df] = compactar_prefijos_columnas(df)

In [37]:
for nombre_df, df in dfs.items():
    # Sacamos la estación de cada fecha
    df['Date'] = pd.to_datetime(df['Date'])
    def get_season(month):
        if month in [12, 1, 2]:
            return 'Invierno'
        elif month in [3, 4, 5]:
            return 'Primavera'
        elif month in [6, 7, 8]:
            return 'Verano'
        else:
            return 'Otoño'
    df['Season'] = df['Date'].dt.month.apply(get_season)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')
    df = pd.concat([df.drop(columns=["Season"]),pd.get_dummies(df["Season"])], axis=1)
    dfs[nombre_df] = df

In [40]:
dfs["df_tifs_C2X-Complex_rhow_5x5"].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 59 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   Date                               36 non-null     datetime64[ns]
 1   Buoy                               36 non-null     category      
 2   Latitude                           36 non-null     int64         
 3   Longitude                          36 non-null     int64         
 4   rhow_B1                            36 non-null     float64       
 5   rhow_B2                            36 non-null     float64       
 6   rhow_B3                            36 non-null     float64       
 7   rhow_B4                            36 non-null     float64       
 8   rhow_B5                            36 non-null     float64       
 9   rhow_B6                            36 non-null     float64       
 10  rhow_B7                            36 no

## Aplicar modelos

In [42]:
import os
import json
import re
import numpy as np
import pandas as pd
from typing import Optional

from joblib import load as joblib_load

# Cargadores específicos (solo si los necesitas)
try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor = None


def _stem_and_keys(fname: str):
    """
    Obtiene el 'stem' base (sin sufijo) y separa dataset y modelo.
    E.g. 'TOA_9x9_depth_in_2_3_KNN_model.joblib' ->
         stem='TOA_9x9_depth_in_2_3_KNN', dataset='TOA_9x9_depth_in_2_3', model='KNN'
    """
    stem = (fname
            .replace("_model.joblib", "")
            .replace("_model.json", "")
            .replace("_model.cbm", "")
            .replace("_features.json", "")
            .replace("_metadata.json", ""))
    # dataset = todo menos el último bloque; model = último bloque
    parts = stem.split("_")
    model = parts[-1]
    dataset = "_".join(parts[:-1]) if len(parts) > 1 else stem
    return stem, dataset, model


def discover_artifacts(models_dir: str):
    """
    Explora la carpeta y devuelve un dict:
    artifacts[dataset][model] = {'model_path': ..., 'features_path': ..., 'metadata_path': ..., 'format': ...}
    """
    artifacts = {}
    for fname in os.listdir(models_dir):
        if not any(fname.endswith(suf) for suf in ["_model.joblib", "_model.json", "_model.cbm",
                                                   "_features.json", "_metadata.json"]):
            continue

        stem, dataset, model = _stem_and_keys(fname)
        artifacts.setdefault(dataset, {}).setdefault(model, {})
        entry = artifacts[dataset][model]

        fpath = os.path.join(models_dir, fname)
        if fname.endswith("_model.joblib"):
            entry["model_path"] = fpath
            entry["format"] = "joblib"
        elif fname.endswith("_model.json"):
            entry["model_path_json"] = fpath
            entry["format"] = entry.get("format", "json")
        elif fname.endswith("_model.cbm"):
            entry["model_path_cbm"] = fpath
            entry["format"] = "cbm"
        elif fname.endswith("_features.json"):
            entry["features_path"] = fpath
        elif fname.endswith("_metadata.json"):
            entry["metadata_path"] = fpath

    return artifacts


def load_features_list(features_path: str):
    """
    Lee *_features.json. Se espera una lista de nombres de columnas o un dict con la clave 'features'.
    """
    with open(features_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        # admite tanto {'features': [...]} como {'feature_names': [...]}
        for k in ("features", "feature_names", "columns"):
            if k in data and isinstance(data[k], list):
                return data[k]
        # si viniese otra estructura, último recurso: llaves del dict si parecen nombres
        return list(data.keys())
    elif isinstance(data, list):
        return data
    else:
        raise ValueError(f"Formato de features no soportado en {features_path}")


def load_model_for_entry(dataset: str, model_name: str, entry: dict):
    """
    Carga el modelo según el formato disponible en 'entry'.
    - Preferencia: joblib (suele incluir Pipeline)
    - XGBoost JSON: crea un XGBRegressor y load_model(json_path)
    - CatBoost CBM: CatBoostRegressor().load_model(cbm_path)
    """
    # 1) Joblib (scikit-learn / pipeline)
    if "model_path" in entry and entry.get("format") == "joblib":
        return joblib_load(entry["model_path"])

    # 2) XGBoost JSON
    if entry.get("format") == "json" and "model_path_json" in entry:
        if XGBRegressor is None:
            raise ImportError("xgboost no está disponible en el entorno.")
        model = XGBRegressor()
        model.load_model(entry["model_path_json"])
        return model

    # 3) CatBoost CBM
    if entry.get("format") == "cbm" and "model_path_cbm" in entry:
        if CatBoostRegressor is None:
            raise ImportError("catboost no está disponible en el entorno.")
        model = CatBoostRegressor(verbose=False)
        model.load_model(entry["model_path_cbm"])
        return model

    raise FileNotFoundError(f"No se pudo cargar modelo para {dataset}/{model_name}: {entry}")


def apply_models(df: pd.DataFrame,
                 models_dir: str,
                 clip_min: Optional[float] = None,
                 strict: bool = False) -> pd.DataFrame:
    """
    Aplica todos los modelos encontrados en 'models_dir' sobre 'df'.
    - clip_min: si se indica, hace clip mínimo (p.ej. 0.3) a las predicciones.
    - strict: si True, exige que existan TODAS las columnas requeridas; si False, avisa y salta el modelo.

    Devuelve un DataFrame con columnas '<dataset>__<modelo>'.
    """
    artifacts = discover_artifacts(models_dir)
    preds = {}

    for dataset, models in artifacts.items():
        for model_name, entry in models.items():
            # cargar lista de features
            if "features_path" not in entry:
                print(f"[AVISO] Sin features.json para {dataset}/{model_name}. "
                      f"Intento inferir features del modelo si es posible...")
                features = None
            else:
                features = load_features_list(entry["features_path"])

            # cargar modelo
            try:
                model = load_model_for_entry(dataset, model_name, entry)
            except Exception as e:
                print(f"[ERROR] No se pudo cargar {dataset}/{model_name}: {e}")
                continue

            # intentar inferir features si no hay json
            if features is None:
                if hasattr(model, "feature_names_in_"):
                    features = list(model.feature_names_in_)
                else:
                    print(f"[AVISO] No se pueden inferir features para {dataset}/{model_name}. Saltando.")
                    continue

            # verificar columnas disponibles
            missing = [c for c in features if c not in df.columns]
            if missing:
                msg = (f"[{'ERROR' if strict else 'AVISO'}] Faltan {len(missing)} columnas "
                       f"en {dataset}/{model_name}: {missing[:5]}{'...' if len(missing) > 5 else ''}")
                if strict:
                    raise KeyError(msg)
                print(msg)
                continue

            # seleccionar y ordenar
            X = df[features].copy()
            # asegurar numérico cuando aplique
            for col in X.columns:
                if pd.api.types.is_object_dtype(X[col]):
                    # intenta convertir a numérico si procede
                    try:
                        X[col] = pd.to_numeric(X[col])
                    except Exception:
                        pass

            # predecir
            try:
                y_hat = model.predict(X)
            except TypeError:
                # algunos wrappers requieren .values
                y_hat = model.predict(X.values)

            y_hat = np.asarray(y_hat).ravel()
            if clip_min is not None:
                y_hat = np.clip(y_hat, clip_min, None)

            col_name = f"{dataset}__{model_name}"
            preds[col_name] = y_hat

    if not preds:
        raise RuntimeError("No se generaron predicciones (revisa logs de avisos/errores).")

    preds_df = pd.DataFrame(preds, index=df.index)
    return preds_df


In [43]:
dfs["df_tifs_C2X-Complex_rhow_9x9"]

,Date,Buoy,Latitude,Longitude,rhow_B1,rhow_B2,rhow_B3,rhow_B4,rhow_B5,rhow_B6,...,dif_rel_4bands_rhow_B3_B2_B5_B4,dif_rel_4bands_rhow_B3_B4_B5_B2,dif_rel_4bands_rhow_B3_B5_B4_B2,dif_rel_4bands_rhow_B4_B2_B5_B3,dif_rel_4bands_rhow_B4_B3_B5_B2,sum_norm_3bands_rhow_B2_B4_B3,sum_norm_3bands_rhow_B3_B5_B4,Otoño,Primavera,Verano
0,2016-09-08,CTD1,4187246,695025,0.010580,0.015641,0.026116,0.013858,0.012464,0.003613,...,0.770,1.088,1.209,0.409,-0.266,0.706,0.965,True,False,False
1,2016-09-08,CTD2,4181518,693105,0.010295,0.014946,0.024088,0.011801,0.009841,0.002800,...,0.778,1.383,1.658,0.381,-0.169,0.685,0.945,True,False,False
2,2016-09-08,CTD3,4181698,695238,0.010014,0.014341,0.022994,0.011156,0.009591,0.002780,...,0.744,1.392,1.619,0.361,-0.184,0.683,0.954,True,False,False
3,2016-09-08,CTD4,4180266,698264,0.009696,0.013779,0.020961,0.011210,0.009079,0.002640,...,0.711,1.211,1.495,0.380,-0.124,0.719,0.934,True,False,False
4,2016-09-08,CTD5,4179450,700268,0.009379,0.014307,0.019780,0.004109,0.003065,0.000704,...,0.637,4.599,6.165,0.132,-0.007,0.540,0.956,True,False,False
5,2016-09-08,CTD6,4176009,695829,0.008654,0.012531,0.019597,0.010282,0.008222,0.002336,...,0.764,1.250,1.563,0.401,-0.131,0.710,0.931,True,False,False
6,2016-09-08,CTD7,4176724,690397,0.010933,0.016065,0.025683,0.012873,0.010826,0.003091,...,0.758,1.321,1.571,0.380,-0.173,0.693,0.947,True,False,False
7,2016-09-08,CTD8,4174178,693048,0.009821,0.014591,0.024009,0.012915,0.011146,0.003214,...,0.782,1.095,1.269,0.421,-0.226,0.713,0.952,True,False,False
8,2016-09-08,CTD9,4171106,693183,0.011150,0.017611,0.032063,0.018600,0.016469,0.004695,...,0.935,0.789,0.891,0.543,-0.355,0.729,0.958,True,False,False
9,2016-09-08,CTD10,4170388,695646,0.011831,0.017778,0.031085,0.016911,0.014817,0.004297,...,0.872,1.005,1.147,0.475,-0.289,0.710,0.956,True,False,False


In [45]:

# df_in: DataFrame con las columnas necesarias (mismas que en *_features.json)
carpeta_modelos = "training_results/models/"  # donde están los *_model.*, *_features.json, *_metadata.json

preds = apply_models(dfs["df_tifs_C2X-Complex_rhow_9x9"], carpeta_modelos, clip_min=0.2, strict=True)
# 'preds' tendrá columnas como:
# 'TOA_9x9_depth_in_2_3__KNN', 'C2X-Complex_rhow_9x9_depth_in_1_2__CAT', ...



KeyError: "[ERROR] Faltan 1 columnas en C2X-Complex_rhow_9x9_depth_in_1_2/CAT: ['Invierno']"